In [4]:
!pip install transformers sentencepiece torch pandas openpyxl tqdm


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm

In [2]:
input_file = "eng_updated selected dataset.xlsx"

df = pd.read_excel(input_file)
df.head()

,sentence_id,orginal_sen
0,1,Let's try something.
1,2,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!
3,4,Muiriel is 20 now.
4,5,"The password is ""Muiriel""."


In [3]:
english_col = "orginal_sen"   

if "sentence_id" not in df.columns:
    df["sentence_id"] = range(1, len(df) + 1)

df = df[["sentence_id", english_col]].copy()
df.rename(columns={english_col: "en_original"}, inplace=True)

df.head()

,sentence_id,en_original
0,1,Let's try something.
1,2,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!
3,4,Muiriel is 20 now.
4,5,"The password is ""Muiriel""."


In [4]:
en_hi_model_name = "Helsinki-NLP/opus-mt-en-hi"
hi_en_model_name = "Helsinki-NLP/opus-mt-hi-en"

tokenizer_en_hi = MarianTokenizer.from_pretrained(en_hi_model_name)
model_en_hi = MarianMTModel.from_pretrained(en_hi_model_name)

tokenizer_hi_en = MarianTokenizer.from_pretrained(hi_en_model_name)
model_hi_en = MarianMTModel.from_pretrained(hi_en_model_name)

print("Models loaded")

c:\Users\manda\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Models loaded


In [5]:
def translate_text(text, tokenizer, model):
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [6]:
tqdm.pandas()

df["hi_translation"] = df["en_original"].progress_apply(
    lambda x: translate_text(x, tokenizer_en_hi, model_en_hi)
)

df.head()

100%|██████████| 200/200 [01:18<00:00,  2.55it/s]


,sentence_id,en_original,hi_translation
0,1,Let's try something.,चलो कुछ कोशिश करते हैं.
1,2,I have to go to sleep.,मुझे नींद में जाना है.
2,3,Today is June 18th and it is Muiriel's birthday!,आज जून 18 वीं है और यह Marris का जन्मदिन है!
3,4,Muiriel is 20 now.,मुंबरी अब 20 साल की है.
4,5,"The password is ""Muiriel"".","पासवर्ड है ""metel"""


In [7]:
df["en_backtranslation"] = df["hi_translation"].progress_apply(
    lambda x: translate_text(x, tokenizer_hi_en, model_hi_en)
)

df.head()

100%|██████████| 200/200 [01:30<00:00,  2.20it/s]


,sentence_id,en_original,hi_translation,en_backtranslation
0,1,Let's try something.,चलो कुछ कोशिश करते हैं.,Let's try something.
1,2,I have to go to sleep.,मुझे नींद में जाना है.,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!,आज जून 18 वीं है और यह Marris का जन्मदिन है!,Today is June 18th and it's Marky's birthday!
3,4,Muiriel is 20 now.,मुंबरी अब 20 साल की है.,Muller is now 20 years old.
4,5,"The password is ""Muiriel"".","पासवर्ड है ""metel""","Password is ""mel"""


In [8]:
df[["sentence_id", "en_original", "hi_translation", "en_backtranslation"]].head(10)

,sentence_id,en_original,hi_translation,en_backtranslation
0,1,Let's try something.,चलो कुछ कोशिश करते हैं.,Let's try something.
1,2,I have to go to sleep.,मुझे नींद में जाना है.,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!,आज जून 18 वीं है और यह Marris का जन्मदिन है!,Today is June 18th and it's Marky's birthday!
3,4,Muiriel is 20 now.,मुंबरी अब 20 साल की है.,Muller is now 20 years old.
4,5,"The password is ""Muiriel"".","पासवर्ड है ""metel""","Password is ""mel"""
5,6,I will be back soon.,मैं जल्द ही वापस हो जाएगा.,I'll be back soon.
6,7,I'm at a loss for words.,मैं शब्दों के लिए एक नुकसान में हूँ.,I'm in a loss for words.
7,8,This is never going to end.,यह कभी अंत करने के लिए नहीं जा रहा है.,It's never going to end.
8,9,I just don't know what to say.,मैं सिर्फ क्या कहना है पता नहीं है.,I don't know just what to say.
9,10,That was an evil bunny.,यह एक बुरा बकवास था.,It was a bad shit.


In [10]:
df.to_excel("marian_bt.xlsx", index=False)
print("Saved marian_bt.xlsx")

Saved marian_bt.xlsx
